1. Frame Extraction

In [ ]:
import cv2
import os

def extract_frames(video_path, output_folder, frames_per_second):
    if not os.path.isfile(video_path):
        print(f"Error: Video file not found at {video_path}")
        return
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Unable to open video file {video_path}")
        return
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    print(f"Video FPS: {fps}")
    
    frame_interval = fps // frames_per_second
    print(f"Frame Interval: {frame_interval}")
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break

        if frame_count % frame_interval == 0:

            output_file = os.path.join(output_folder, f"vid_{frame_count // frame_interval}.jpg")
            print(f"Saving frame {frame_count // frame_interval} to {output_file}")
            
            cv2.imwrite(output_file, frame)
        
        frame_count += 1
    
    cap.release()
    print("Frame extraction completed.")

video_path = "..."
output_folder = "..."
frames_per_second = 2

os.makedirs(output_folder, exist_ok=True)
extract_frames(video_path, output_folder, frames_per_second)


2. Dataset Seperation

In [ ]:
import os
import random
import shutil

dataset_dir = r"..." 
output_dir = r"..." 

splits = ["train", "test"]
for split in splits:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

images = [f for f in os.listdir(dataset_dir) if os.path.isfile(os.path.join(dataset_dir, f))]

random.shuffle(images)

total = len(images)
train_size = int(0.7 * total)
test_size = total - train_size

train_files = images[:train_size]
test_files = images[train_size:]

def copy_files(file_list, dest_folder):
    for file in file_list:
        src = os.path.join(dataset_dir, file)
        dst = os.path.join(output_dir, dest_folder, file)
        shutil.copy(src, dst)

copy_files(train_files, "train")
copy_files(test_files, "test")

print(f"Total images: {total}")
print(f"Train: {len(train_files)}")
print(f"Test:  {len(test_files)}")

3. Training Model

In [ ]:
import os, json, random
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

import segmentation_models_pytorch as smp

import seaborn as sns
import matplotlib.pyplot as plt

TRAIN_IMAGES = r"..."
TRAIN_LABELS = r"..."
TEST_IMAGES  = r"..."
TEST_LABELS  = r"..."

LABEL_NAME_TO_ID = {
    "background": 0,
    "encrusting": 1,
    "plate": 2,
    "massive": 3,
    "folios": 4,
    "branching": 5,
}

MINORITY_CLASSES = [4, 5]

NUM_CLASSES = 6
IGNORE_INDEX = 255

NUM_EPOCHS = 60
BATCH_SIZE = 8
LR = 1e-4
INPUT_SIZE = (512, 512)

def normalize_label(label):
    return label.lower().strip().replace(" ", "").replace("-", "")


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class LabelMeDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def load_mask(self, json_path, size):
        with open(json_path) as f:
            data = json.load(f)

        mask = Image.new("L", size, 0)
        draw = ImageDraw.Draw(mask)

        for shp in data["shapes"]:
            raw_label = shp["label"]
            label = normalize_label(raw_label)

            if label not in LABEL_NAME_TO_ID:
                continue

            cls = LABEL_NAME_TO_ID[label]

            pts = shp["points"]
            draw.polygon([tuple(p) for p in pts], fill=cls)

        return mask

    def has_minority(self, mask):
        arr = np.array(mask)
        return any((arr == c).any() for c in MINORITY_CLASSES)

    def augment_fn(self, img, mask):
        if random.random() < 0.5:
            img = TF.hflip(img); mask = TF.hflip(mask)

        if random.random() < 0.3:
            img = TF.vflip(img); mask = TF.vflip(mask)

        if self.has_minority(mask):
            if random.random() < 0.5:
                angle = random.randint(-30, 30)
                img = TF.rotate(img, angle)
                mask = TF.rotate(mask, angle)

            if random.random() < 0.4:
                img = TF.adjust_brightness(img, random.uniform(0.7, 1.3))
                img = TF.adjust_contrast(img, random.uniform(0.7, 1.3))

        return img, mask

    def __getitem__(self, idx):
        img_path, json_path = self.pairs[idx]

        img = Image.open(img_path).convert("RGB")
        mask = self.load_mask(json_path, img.size)

        img = img.resize(INPUT_SIZE[::-1], resample=Image.BILINEAR)
        mask = mask.resize(INPUT_SIZE[::-1], resample=Image.NEAREST)

        if self.augment:
            img, mask = self.augment_fn(img, mask)

        img = TF.to_tensor(img)
        mask = torch.from_numpy(np.array(mask)).long()

        return img, mask

def get_pairs(img_dir, label_dir):
    pairs = []
    for img in Path(img_dir).glob("*"):
        json_path = Path(label_dir) / f"{img.stem}.json"
        if json_path.exists():
            pairs.append((img, json_path))
    return pairs


def compute_metrics(preds, labels, num_classes):
    preds = preds.view(-1)
    labels = labels.view(-1)

    mask = labels != IGNORE_INDEX
    preds = preds[mask]
    labels = labels[mask]

    confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    for t, p in zip(labels, preds):
        confusion[t.long(), p.long()] += 1

    accuracy = torch.diag(confusion).sum().float() / confusion.sum().float()

    iou_per_class = []
    for i in range(num_classes):
        TP = confusion[i, i]
        FP = confusion[:, i].sum() - TP
        FN = confusion[i, :].sum() - TP

        denom = TP + FP + FN
        iou = TP.float() / denom.float() if denom != 0 else torch.tensor(0.0)
        iou_per_class.append(iou)

    miou = torch.mean(torch.stack(iou_per_class))

    return accuracy.item(), miou.item(), confusion


def plot_confusion_matrix(cm):
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=LABEL_NAME_TO_ID.keys(),
        yticklabels=LABEL_NAME_TO_ID.keys()
    )

    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title("Best Confusion Matrix")

    plt.savefig("best_confusion_matrix.png")
    plt.close()

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_pairs = get_pairs(TRAIN_IMAGES, TRAIN_LABELS)
    val_pairs   = get_pairs(TEST_IMAGES, TEST_LABELS)

    train_ds = LabelMeDataset(train_pairs, augment=True)
    val_ds   = LabelMeDataset(val_pairs, augment=False)

    train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds, BATCH_SIZE)

    model = smp.DeepLabV3Plus(
        encoder_name="resnet101",
        encoder_weights="imagenet",
        classes=NUM_CLASSES
    ).to(device)

    weights = torch.tensor([1.0,1.0,1.0,1.0,2.0,2.0]).to(device)

    criterion = FocalLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_miou = 0
    patience = 10
    no_improve = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0

        for imgs, masks in train_dl:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_dl)

        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs).argmax(1)

                all_preds.append(preds.cpu())
                all_labels.append(masks.cpu())

        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)

        acc, miou, confusion = compute_metrics(all_preds, all_labels, NUM_CLASSES)

        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        print(f"Train Loss     : {avg_loss:.4f}")
        print(f"Nilai Accuracy : {acc:.4f}")
        print(f"Nilai mIoU     : {miou:.4f}")

        if miou > best_miou:
            best_miou = miou
            no_improve = 0

            torch.save(model.state_dict(), "deeplabv3plus_resnet101-07.pt")

            plot_confusion_matrix(confusion.numpy())

            print("Best model saved")

        else:
            no_improve += 1

    print("\nTraining selesai")

if __name__ == "__main__":
    main()

4. Coral Detection

In [ ]:
import os
import math
import numpy as np
from PIL import Image

import cv2

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp

from tqdm import tqdm

MODEL_PATH = r"C:\Workspace\TEEP\ModelTraining\Program\checkpoints\deeplabv3plus_resnet101-07.pt"

ORTHO_IMAGE = r"C:\Workspace\TEEP\ModelTraining\Program\Test\Orthomosaic.png"

OUTPUT_MASK = "pred_mask.png"
OUTPUT_COLOR = "pred_color.png"
OUTPUT_OVERLAY = "pred_overlay.png"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 6
TILE_SIZE = 512
OVERLAP = 128
INPUT_SIZE = (512, 512)
ALPHA = 0.5
USE_MORPHOLOGY = True
MORPH_KERNEL = 3

CLASS_WEIGHTS = [
    0.85,   # background
    0.75,   # encrusting
    1.15,   # plate
    1.30,   # massive
    1.35,   # folios
    1.35    # branching
]

BASE_COLORS = [
    (0, 0, 0),        # background
    (240, 0, 20),     # encrusting
    (0, 100, 255),    # plate
    (255, 255, 0),    # massive
    (138, 43, 226),   # folios
    (50, 75, 255)     # branching
]

print("Loading model...")

model = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights=None,
    classes=NUM_CLASSES
)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)

model.to(DEVICE)
model.eval()

print("Model loaded")
print("\nLoading orthomosaic...")
ortho = Image.open(ORTHO_IMAGE).convert("RGB")
W, H = ortho.size
print(f"Image size : {W} x {H}")
step = TILE_SIZE - OVERLAP
x_positions = list(range(0, W, step))
y_positions = list(range(0, H, step))
total_tiles = len(x_positions) * len(y_positions)
print(f"Total tiles : {total_tiles}")
prob_map = np.zeros(
    (NUM_CLASSES, H, W),
    dtype=np.float32
)
weight_map = np.zeros(
    (H, W),
    dtype=np.float32
)
yy, xx = np.mgrid[0:TILE_SIZE, 0:TILE_SIZE]
center_y = TILE_SIZE / 2
center_x = TILE_SIZE / 2
distance = np.sqrt(
    (xx - center_x) ** 2 +
    (yy - center_y) ** 2
)
distance = distance / distance.max()
tile_weight = 1.0 - distance
tile_weight = np.clip(tile_weight, 0.05, 1.0)
tile_weight = tile_weight.astype(np.float32)

def predict_with_tta(tensor):
    preds = []
    logits = model(tensor)
    preds.append(
        F.softmax(logits, dim=1)
    )
    flip_h = torch.flip(
        tensor,
        dims=[3]
    )
    logits_h = model(flip_h)
    logits_h = torch.flip(
        logits_h,
        dims=[3]
    )
    preds.append(
        F.softmax(logits_h, dim=1)
    )
    flip_v = torch.flip(
        tensor,
        dims=[2]
    )
    logits_v = model(flip_v)

    logits_v = torch.flip(
        logits_v,
        dims=[2]
    )
    preds.append(
        F.softmax(logits_v, dim=1)
    )
    probs = torch.mean(
        torch.stack(preds),
        dim=0
    )

    return probs

print("\nRunning inference...")

with torch.no_grad():
    pbar = tqdm(total=total_tiles)
    for y in y_positions:
        for x in x_positions:
            x_end = min(x + TILE_SIZE, W)
            y_end = min(y + TILE_SIZE, H)

            tile = ortho.crop(
                (x, y, x_end, y_end)
            )
            original_w = x_end - x
            original_h = y_end - y
            tile_resized = tile.resize(
                INPUT_SIZE,
                resample=Image.BILINEAR
            )
            tensor = TF.to_tensor(
                tile_resized
            ).unsqueeze(0).to(DEVICE)
            probs = predict_with_tta(tensor)
            probs = probs.squeeze(0).cpu().numpy()
            resized_probs = []
            for c in range(NUM_CLASSES):
                prob_img = Image.fromarray(
                    probs[c]
                )
                prob_img = prob_img.resize(
                    (original_w, original_h),
                    resample=Image.BILINEAR
                )
                resized_probs.append(
                    np.array(prob_img)
                )
            probs = np.stack(resized_probs)
            local_weight = tile_weight[
                :original_h,
                :original_w
            ]
            for c in range(NUM_CLASSES):

                prob_map[
                    c,
                    y:y_end,
                    x:x_end
                ] += (
                    probs[c] * local_weight
                )
            weight_map[
                y:y_end,
                x:x_end
            ] += local_weight
            pbar.update(1)
    pbar.close()

print("\nNormalizing probability map...")

prob_map /= np.maximum(
    weight_map,
    1e-6
)

print("Applying class balancing...")

for class_id in range(NUM_CLASSES):

    prob_map[class_id] *= CLASS_WEIGHTS[class_id]

print("Generating final mask...")

final_mask = np.argmax(
    prob_map,
    axis=0
).astype(np.uint8)

if USE_MORPHOLOGY:

    print("Applying morphology refinement...")

    kernel = np.ones(
        (MORPH_KERNEL, MORPH_KERNEL),
        np.uint8
    )

    refined_mask = np.zeros_like(final_mask)

    for cls in range(NUM_CLASSES):

        binary = (
            final_mask == cls
        ).astype(np.uint8)

        binary = cv2.morphologyEx(
            binary,
            cv2.MORPH_CLOSE,
            kernel
        )

        refined_mask[binary == 1] = cls

    final_mask = refined_mask

Image.fromarray(final_mask).save(
    OUTPUT_MASK
)
print("Mask grayscale saved")
print("Creating color mask...")
color_mask = np.zeros(
    (H, W, 3),
    dtype=np.uint8
)
for class_id, color in enumerate(BASE_COLORS):
    color_mask[
        final_mask == class_id
    ] = color
Image.fromarray(color_mask).save(
    OUTPUT_COLOR
)
print("Color mask saved")
print("Creating overlay...")
ortho_np = np.array(ortho)
overlay = (
    ortho_np * (1 - ALPHA) +
    color_mask * ALPHA
).astype(np.uint8)
Image.fromarray(overlay).save(
    OUTPUT_OVERLAY
)
print("Overlay saved")
print("Inference selesai")

print(f"Mask     : {OUTPUT_MASK}")
print(f"Color    : {OUTPUT_COLOR}")
print(f"Overlay  : {OUTPUT_OVERLAY}")

5. Coverage Calculation

In [ ]:
import csv
from pathlib import Path
import numpy as np
from PIL import Image

MASK_PATH = r"C:\Workspace\TEEP\ModelTraining\Program\Test\gt_mask.png"
CSV_PATH = r"C:\Workspace\TEEP\ModelTraining\Program\coral_area_m2.csv"

PIXEL_SIZE_M = 0.00166

CLASS_NAMES = [
    "background",
    "encrusting",
    "plate",
    "massive",
    "folios",
    "branching"
]

def main():
    print("Loading mask")
    mask = np.array(Image.open(MASK_PATH))
    coral_mask = mask != 0
    total_coral_pixels = np.sum(coral_mask)
    pixel_area_m2 = PIXEL_SIZE_M ** 2
    results = []
    print("\nCORAL AREA RESULT")
    for class_id in range(1, len(CLASS_NAMES)):

        class_name = CLASS_NAMES[class_id]
        class_pixels = np.sum(mask == class_id)
        class_area_m2 = (
            class_pixels * pixel_area_m2
        )
        coverage_percent = (
            (class_pixels / total_coral_pixels) * 100
            if total_coral_pixels > 0 else 0
        )
        results.append([
            class_name,
            int(class_pixels),
            round(class_area_m2, 6),
            round(coverage_percent, 3)
        ])
        print(
            f"{class_name:<12} | "
            f"Pixels: {class_pixels:<10} | "
            f"Area: {class_area_m2:.6f} m² | "
            f"Coverage: {coverage_percent:.3f}%"
        )
    total_area_m2 = (
        total_coral_pixels * pixel_area_m2
    )
    print(f"Total Coral Pixels : {total_coral_pixels}")
    print(f"Total Coral Area   : {total_area_m2:.6f} m²")
    print("Total Coverage     : 100.000%")

    Path(CSV_PATH).parent.mkdir(
        parents=True,
        exist_ok=True
    )
    with open(CSV_PATH, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Class",
            "Pixels",
            "Area_m2",
            "Coverage_Percent"
        ])
        writer.writerows(results)
        writer.writerow([])
        writer.writerow([
            "TOTAL_CORAL",
            int(total_coral_pixels),
            round(total_area_m2, 6),
            100.000
        ])
    print(f"\nCSV saved : {CSV_PATH}")

if __name__ == "__main__":
    main()